In [0]:
dbutils.widgets.text("PromotionName", "4P Beta")
PromotionName = dbutils.widgets.get("PromotionName")

dbutils.widgets.text("job_id","-1")
JobId = dbutils.widgets.get("job_id")

dbutils.widgets.text("run_id","-1")
PipelineRunId = dbutils.widgets.get("run_id")

ConfigID=dbutils.widgets.text("ConfigID","52")
ConfigID=dbutils.widgets.get("ConfigID")

dbutils.widgets.text('sourceTypeId','15')
sourceTypeId = dbutils.widgets.get('sourceTypeId')

dbutils.widgets.text('CreatedBy','ADB_ProcessCosumer_InvoiceAddendumDetails')
CreatedBy = dbutils.widgets.get('CreatedBy')

dbutils.widgets.text('UpdatedBy','ADB_ProcessCosumer_InvoiceAddendumDetails')
UpdatedBy = dbutils.widgets.get('UpdatedBy')

dbutils.widgets.text('silverpath_UlLogs','/mnt/silver/FACTULLogs')
silverpath_UlLogs = dbutils.widgets.get('silverpath_UlLogs')

dbutils.widgets.text('silverpath_InvoiceAddendumDetails','/mnt/silver/FACTInvoiceAddendumDetails')
silverpath_InvoiceAddendumDetails = dbutils.widgets.get('silverpath_InvoiceAddendumDetails')

dbutils.widgets.text('silverpath_RulesEngineAuditLog','/mnt/silver/FACTRulesEngineAuditLog')
silverpath_RulesEngineAuditLog = dbutils.widgets.get('silverpath_RulesEngineAuditLog')

checkPointLocation = "/_checkpoints/"

In [0]:
%run /Configurations/Init_Scripts

In [0]:
%run ./PopulateComments_GenericFunctions

In [0]:
%run ./Process_InvoiceException

In [0]:
%run ./Process_ConsumerSubscription_Live

In [0]:
%run ./Process_InvoiceAddendumDetails_Live

In [0]:
%run ./Process_ConsumerSubscription_TruUp

In [0]:
%run ./Process_InvoiceAddendumDetails_TruUp

In [0]:
def processCycleData(df_CycleData,UpdatedBy):
    df_FACTInvoiceAddendum = (spark.read.table('Promotion.FACT_InvoiceAddendum')
                                        .groupBy("PromotionUUID")
                                        .agg(max("InvoiceDate").alias("MaxInvoiceDate")))

    df_ValidCycleData = getValidPromotionCycles_Overrides(df_CycleData, PromotionName)

    df_AccountFlags =  getAccountFlags(df_ValidCycleData)
    df_CycleData_AGG = df_ValidCycleData.join(df_AccountFlags,['ShipToAccountUUID','PromotionUUID','CycleDate'],'left')
    
    df_FACTConsumerSubscription_IAD = (spark.read.table('Promotion.FACT_ConsumerSubscription')
                                .withColumn('PilotSubscriptionFlag',coalesce(col('PilotSubscriptionFlag'),lit('N')))
                                .withColumn('lagSubscriptionEndDate',lag("SubscriptionEndDate").over(Window.partitionBy("ShipToAccountUUID", "CoolSculptingID").orderBy(asc("SubscriptionEndDate"))))
                                .withColumn('MaxSubscriptionEndDate',max("SubscriptionEndDate").over(Window.partitionBy("ShipToAccountUUID", "CoolSculptingID")))
                                .withColumn('PreviousSubscriptionEndDate',when(col('lagSubscriptionEndDate').isNull(),
                                                                                date_sub(col('SubscriptionStartDate'), 150))
                                                                            .otherwise(col('lagSubscriptionEndDate')))
                                .filter("VersionId = 1 AND SubscriptionEndDate >= date_sub(current_date(),300)")
                    ) 
    
    df_CycleData_Consumer = (df_CycleData_AGG.join(df_FACTConsumerSubscription_IAD,
                                         (df_CycleData_AGG.CoolSculptingID == df_FACTConsumerSubscription_IAD.CoolSculptingID) &
                                         ((df_CycleData_AGG.ShipToAccountUUID == df_FACTConsumerSubscription_IAD.ShipToAccountUUID) 
                                          |
                                          ((df_CycleData_AGG.SoldToAccountID == df_FACTConsumerSubscription_IAD.SoldToAccountID) &
                                           (df_CycleData_AGG.ShipToAccountUUID != df_FACTConsumerSubscription_IAD.ShipToAccountUUID))
                                         )&
                                         (df_CycleData_AGG.PromotionUUID == df_FACTConsumerSubscription_IAD.PromotionUUID) &
                                         (df_CycleData_AGG.CycleDate.between(df_FACTConsumerSubscription_IAD.PreviousSubscriptionEndDate,
                                                                             df_FACTConsumerSubscription_IAD.SubscriptionEndDate))
                                         ,
                                         "left")
                                        .select(
                                                df_CycleData_AGG.CycleDataUUID,
                                                df_CycleData_AGG.CoolSculptingID,
                                                df_CycleData_AGG.ShipToAccountUUID,
                                                df_CycleData_AGG.ShipToAccountID,
                                                df_CycleData_AGG.PromotionUUID,
                                                df_CycleData_AGG.CycleDate,
                                                df_CycleData_AGG.MinCycleTmstmp,
                                                df_CycleData_AGG.CycleCount,
                                                df_CycleData_AGG.SoldToAccountID,
                                                df_CycleData_AGG.FlagType,
                                                df_CycleData_AGG.FlagSubType,
                                                df_FACTConsumerSubscription_IAD.ConsumerSubscriptionUUID,
                                                df_FACTConsumerSubscription_IAD.SubscriptionStartDate,
                                                df_FACTConsumerSubscription_IAD.SubscriptionEndDate,
                                                df_FACTConsumerSubscription_IAD.VersionID,
                                                df_FACTConsumerSubscription_IAD.Comments,
                                                df_FACTConsumerSubscription_IAD.PilotSubscriptionFlag,
                                                df_FACTConsumerSubscription_IAD.PreviousSubscriptionEndDate,
                                                df_FACTConsumerSubscription_IAD.MaxSubscriptionEndDate,
                                                df_FACTConsumerSubscription_IAD.InvoiceExceptionFlag
                                                )
                                        .fillna('N', subset=['InvoiceExceptionFlag'])
                                        .withColumn('CycleDataDedupFlag',row_number().over(Window.partitionBy('CycleDataUUID')
                                                                                     .orderBy(asc("PilotSubscriptionFlag"))))
                                        .filter('CycleDataDedupFlag = 1')
    )
    
    
    df_CycleData_Consumer = df_CycleData_Consumer.join(df_FACTInvoiceAddendum,["PromotionUUID"],'left')

    df_CycleData_Consumer_CycleFlag = applyRules(df_CycleData_Consumer,'CycleFlag')

    df_CycleData_Consumer_CycleFlag.persist()

    df_CycleData_ConsumerSubscription_Live = df_CycleData_Consumer_CycleFlag.filter('AdjustSubscriptionFlag = 0 AND PerCycleFlag = 0')
    df_CycleData_InvoiceaddendumDetails_Live = df_CycleData_Consumer_CycleFlag.filter('AdjustSubscriptionFlag = 0 OR  PerCycleFlag = 1')
    df_CycleData_ConsumerSubscription_Adjust = df_CycleData_Consumer_CycleFlag.filter('AdjustSubscriptionFlag = 1 AND PerCycleFlag = 0')
    df_CycleData_InvoiceaddendumDetails_Adjust = df_CycleData_Consumer_CycleFlag.filter('PerCycleFlag = 0')

    if(df_CycleData_ConsumerSubscription_Adjust.count()>0):
        processLateCycleSubscriptionAdjustmentsTruUp(df_CycleData_ConsumerSubscription_Adjust,UpdatedBy)
        processLateCycleSubscriptionReallignmentTruUp(df_CycleData_InvoiceaddendumDetails_Adjust,UpdatedBy)
        processInvoiceAddendumDetails_LateCycleTruUp(df_CycleData_InvoiceaddendumDetails_Adjust,UpdatedBy)

    if(df_CycleData_ConsumerSubscription_Live.count()>0):
        populateConsumerSubscription(df_CycleData_ConsumerSubscription_Live,UpdatedBy)

    if(df_CycleData_InvoiceaddendumDetails_Live.count()>0):    
        processInvoiceAddendumDetails(df_CycleData_InvoiceaddendumDetails_Live,UpdatedBy)

    populateGenericComments(UpdatedBy)

    print(df_CycleData.count())
    print(df_CycleData_Consumer_CycleFlag.count())
    df_CycleData_Consumer_CycleFlag.unpersist()

In [0]:
def setWorkFlowStatus(df_DIMCyclesOverride,WorkFlowStatus,UpdatedBy):
    df_DIMCyclesOverride = (df_DIMCyclesOverride
                            .withColumn('WorkFlowStatus',lit(WorkFlowStatus))
                            .withColumn('UpdatedBy',lit(UpdatedBy))
                            .withColumn('UpdatedDate',lit(current_timestamp()))
                            )
    dl_DIMCyclesOverride = DeltaTable.forName(spark, 'promotion.DIM_CyclesOverride')
    (dl_DIMCyclesOverride.alias("tgt")
            .merge(df_DIMCyclesOverride.alias("src"),
                ("tgt.CycleId = src.CycleId AND tgt.PromotionUUID = src.PromotionUUID AND tgt.CreatedDate = src.CreatedDate "))
            .whenMatchedUpdate(set ={
                "tgt.WorkflowStatus": "src.WorkFlowStatus",
                "tgt.UpdatedBy": "src.UpdatedBy",
                "tgt.UpdatedDate": "src.UpdatedDate",                                
                })
    .execute())


In [0]:
def Process_PromotionULCycles_AdhocDCR(UpdatedBy):
    df_DIMCyclesOverride = spark.read.table('promotion.DIM_CyclesOverride').filter('WorkflowStatus IS NULL')
    if(df_DIMCyclesOverride.count()>0):
        setWorkFlowStatus_CyclesOverride(df_DIMCyclesOverride,'InProgress',UpdatedBy)
        df_DIMCyclesOverride_InProgress = spark.read.table('promotion.DIM_CyclesOverride').filter('WorkflowStatus = "InProgress"')
        try:
            processCycleData(df_DIMCyclesOverride_InProgress,UpdatedBy)
            setWorkFlowStatus_CyclesOverride(df_DIMCyclesOverride_InProgress,'Success',UpdatedBy)
        except Exception as e:
            setWorkFlowStatus_CyclesOverride(df_DIMCyclesOverride_InProgress,'Fail',UpdatedBy)
            raise e
    else:
        print('No records to process')


In [0]:
Process_PromotionULCycles_AdhocDCR(UpdatedBy)